# **Partie Maxime**  
Choses à réaliser
- Car counting (instant counting and cumulative counting)
- Congestion detection (sparse, level of congestion light,med,heavy)

Je pars des résultats de KC qui a pu annoter les vidéos

In [ ]:
## Importation des bibliothèques nécessaires
import os

# **Partie 1:** Comptage des voitures sur les vidéos
A partir des résultats de KC, je dois compter le nombre de véhicules présents sur la vidéo sans compter plusieurs fois le même véhicule  
Le comptage doit être fait par type de véhicules détecté dans la vidéo

1. Load the Video  
Use a library like OpenCV to load the video file.
Extract frames from the video for processing.  
2. Vehicle Detection  
Implement a vehicle detection model (e.g., YOLO, SSD) to identify vehicles in each frame.
Ensure the model is trained to recognize different types of vehicles (cars, trucks, buses, etc.).   
3. Track Vehicles  
Use a tracking algorithm (e.g., SORT, Deep SORT) to track detected vehicles across frames.
Assign a unique ID to each detected vehicle to avoid double counting.
4. Count Vehicles by Type  
Maintain a dictionary or similar data structure to count vehicles by type.
For each detected vehicle, check its type and increment the corresponding count.
5. Handle Occlusions  
Implement logic to handle occlusions where vehicles may temporarily disappear from view.
Ensure that vehicles reappearing in subsequent frames are counted only once.
6. Output Results  
After processing the video, output the total counts of each vehicle type.
Optionally, visualize the results by overlaying the counts on the video frames.
7. Save Results  
Save the counts and any relevant data (e.g., vehicle IDs, timestamps) to a file for further analysis.
8. Test and Validate  
Test the implementation on various video samples to ensure accuracy.
Validate the results against ground truth data if available.

# Partie 1: Comptage des voitures sur les vidéos

## Objectif
À partir des résultats de KC (détections YOLO), nous allons compter le nombre de véhicules présents dans chaque vidéo **sans compter plusieurs fois le même véhicule**. Le comptage sera réalisé **par type de véhicules** détectés (car, bus, van).

## Données disponibles

### Après "antoine prepare the data":
- **Vidéos annotées** : `data_processed/train/annotated_videos/` et `data_processed/test/annotated_videos/`
- **Images par séquence** : Structure en dossiers (ex: `MVI_20011/img00001.jpg`)
- **Labels YOLO** : Fichiers `.txt` avec format: `class_id x_center y_center width height`
- **Classes**: 0=car, 1=bus, 2=van

### Après "import_data":
- **Vidéos sources**: `data/video/cctv052x2004080516x01638.avi` et autres vidéos au format `.avi`
- **Fonctions utiles**: `read_video_segment()` pour extraire les frames d'une vidéo

---

## Étapes détaillées

### Étape 1: Charger la vidéo et extraire les frames
- Utiliser `cv2.VideoCapture()` pour charger le fichier vidéo
- Extraire les propriétés: FPS, résolution, nombre total de frames
- Lire tous les frames de la vidéo ou par segments
- Stocker les frames en mémoire ou les traiter par lot

### Étape 2: Charger le modèle YOLO de KC
- Charger le modèle YOLO fine-tuné depuis `vehicle-detection/yolov8n-8classes*/weights/`
- Utiliser le modèle pour faire les prédictions sur chaque frame
- Récupérer les boîtes détectées: `[x, y, width, height, confiance, classe]`
- Filtrer les détections par seuil de confiance (ex: >0.5)

### Étape 3: Implémentation du suivi (Tracking)
**Problème**: Les mêmes véhicules apparaissent dans plusieurs frames
**Solution**: Utiliser un algorithme de suivi pour associer les détections entre frames

Options d'implémentation:
- **Simple (Centroid Tracking)**: Calculer le centroïde de chaque boîte, associer les centroïdes proches d'une frame à la suivante
- **Intermédiaire (SORT)**: Utiliser la librarietorch-sort ou implémenter SORT
- **Avancé (Deep SORT)**: Incorporer des features d'apparence pour améliorer le suivi

Étapes:
```
Pour chaque frame i:
  - Détecter les véhicules (boîtes)
  - Calculer les centroïdes des boîtes détectées
  
  Si c'est la première frame:
    - Attribuer un ID unique à chaque véhicule
  Sinon:
    - Comparer les centroïdes actuels avec les centroïdes de la frame précédente
    - Associer chaque détection actuelle à l'ID du véhicule le plus proche (dans une distance donnée)
    - Pour les nouveaux véhicules non associés, créer un nouvel ID
    - Marquer les véhicules anciens qui n'ont pas d'association comme "disparus" (peut réapparaître)
```

### Étape 4: Gestion des occlusions et disparitions
- **Occlusion**: Un véhicule disparaît temporairement (caché par un autre)
  - Solution: Garder les IDs "disparus" actifs pendant N frames consécutifs
  - Si le véhicule réapparaît à proximité, le réassocier au même ID
  
- **Disparition définitive**: Un véhicule quitte le champ de vision
  - Solution: Après N frames sans détection, marquer l'ID comme "complet" et l'ajouter au comptage final

### Étape 5: Compter les véhicules par type
Maintenir un dictionnaire avec la structure:
```python
vehicle_counts = {
    'car': 0,
    'bus': 0,
    'van': 0
}

tracked_vehicles = {
    'vehicle_id_1': {'type': 'car', 'first_frame': 50, 'last_frame': 200, 'frames': [...] },
    'vehicle_id_2': {'type': 'bus', 'first_frame': 10, 'last_frame': 150, 'frames': [...] },
    ...
}
```

Pour chaque véhicule complètement suivi:
- Récupérer le type (classe YOLO)
- Incrémenter le compteur correspondant
- Stocker les informations: ID, type, frame d'apparition, frame de disparition

### Étape 6: Afficher les résultats
- **Statistiques globales**: Nombre total par type
- **Visualisation**: Vidéo avec les boîtes + IDs + classes de chaque véhicule
- **Graphiques**: Histogrammes du nombre de véhicules détectés par type
- **Tableau**: Détails de chaque véhicule suivi (ID, type, durée, position moyenne)

### Étape 7: Validation et gérer les cas spéciaux
- **Véhicules proches**: Deux véhicules très proches peuvent être confondus
  - Solution: Vérifier la taille des boîtes, la continuité du mouvement
  
- **Véhicules stationnés**: Peuvent rester longtemps dans la vidéo
  - Solution: Les compter si présents au moins 1 fois avec confiance suffisante
  
- **Faux positifs**: Détections incorrectes du modèle
  - Solution: Augmenter le seuil de confiance, filtrer par taille minimale

### Étape 8: Sauvegarder les résultats
```python
results = {
    'video_name': 'cctv052x2004080516x01638.avi',
    'total_frames': 5000,
    'fps': 25,
    'vehicle_counts': {'car': 45, 'bus': 12, 'van': 8},
    'tracked_vehicles': [...],
    'processing_time': 125.5  # en secondes
}
```
- Exporter en JSON ou CSV
- Sauvegarder la vidéo annotée avec les IDs de suivi

In [1]:
import os
from glob import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display
from tqdm import tqdm
from collections import defaultdict
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Vérifier la disponibilité de YOLO
try:
    from ultralytics import YOLO
    print("✅ YOLO disponible")
except ImportError:
    print("⚠️ YOLO non installé. Installation en cours...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'ultralytics'])
    from ultralytics import YOLO
    print("✅ YOLO installé")

⚠️ YOLO non installé. Installation en cours...
WARNING torchvision==0.24 is incompatible with torch==2.5.
Run 'pip install torchvision==0.20' to fix torchvision or 'pip install -U torch torchvision' to update both.
For a full compatibility table see https://github.com/pytorch/vision#installation


RuntimeError: operator torchvision::nms does not exist

In [ ]:
# ============================================================================
# Classe pour le suivi des véhicules (Centroid Tracking)
# ============================================================================

class CentroidTracker:
    """
    Suivi des véhicules basé sur les centroïdes
    Associe les détections d'une frame à la suivante en comparant les distances
    """
    def __init__(self, maxDisappeared=30, maxDistance=50):
        """
        Args:
            maxDisappeared: Nombre de frames avant de considérer un véhicule comme disparu
            maxDistance: Distance maximale pour associer une détection actuelle à un ID
        """
        self.nextObjectID = 0
        self.objects = {}  # {ID: centroid}
        self.disappeared = {}  # {ID: nb frames disparus}
        self.vehicle_types = {}  # {ID: class_name}
        self.vehicle_history = defaultdict(list)  # {ID: [(frame_num, bbox, class), ...]}
        self.maxDisappeared = maxDisappeared
        self.maxDistance = maxDistance

    def register(self, centroid, class_name):
        """Enregistrer un nouvel objet"""
        self.objects[self.nextObjectID] = centroid
        self.disappeared[self.nextObjectID] = 0
        self.vehicle_types[self.nextObjectID] = class_name
        self.nextObjectID += 1

    def deregister(self, objectID):
        """Supprimer un objet"""
        del self.objects[objectID]
        del self.disappeared[objectID]
        del self.vehicle_types[objectID]

    def update(self, rects, class_names, frame_num):
        """
        Mettre à jour le suivi avec les nouvelles détections
        
        Args:
            rects: Liste de [(x1, y1, x2, y2), ...] - boîtes détectées
            class_names: Liste des classes correspondantes
            frame_num: Numéro du frame actuel
        
        Returns:
            Dict {ID: centroid} des objets suivis dans ce frame
        """
        if len(rects) == 0:
            # Pas de détection - incrémenter les compteurs d'absence
            disappeared_ids = list(self.disappeared.keys())
            for objectID in disappeared_ids:
                self.disappeared[objectID] += 1
                if self.disappeared[objectID] > self.maxDisappeared:
                    self.deregister(objectID)
            return self.objects

        # Calculer les centroïdes des nouvelles détections
        input_centroids = np.zeros((len(rects), 2))
        for (i, (x1, y1, x2, y2)) in enumerate(rects):
            cX = (x1 + x2) // 2
            cY = (y1 + y2) // 2
            input_centroids[i] = [cX, cY]

        # Si pas d'objet suivis, enregistrer tous les nouveaux
        if len(self.objects) == 0:
            for i in range(0, len(input_centroids)):
                self.register(input_centroids[i], class_names[i])
                bbox = rects[i]
                self.vehicle_history[self.nextObjectID - 1].append({
                    'frame': frame_num,
                    'bbox': bbox,
                    'class': class_names[i]
                })
        else:
            # Associer les détections actuelles aux objets existants
            objectIDs = list(self.objects.keys())
            objectCentroids = list(self.objects.values())

            # Calculer les distances entre centroïdes
            D = np.zeros((len(objectCentroids), len(input_centroids)))
            for i in range(len(objectCentroids)):
                for j in range(len(input_centroids)):
                    D[i, j] = np.linalg.norm(objectCentroids[i] - input_centroids[j])

            # Associer greedily (plus proche match)
            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]

            used_rows = set()
            used_cols = set()

            for (row, col) in zip(rows, cols):
                if row in used_rows or col in used_cols:
                    continue
                if D[row, col] > self.maxDistance:
                    continue

                objectID = objectIDs[row]
                self.objects[objectID] = input_centroids[col]
                self.disappeared[objectID] = 0
                self.vehicle_types[objectID] = class_names[col]
                
                # Ajouter à l'historique
                bbox = rects[col]
                self.vehicle_history[objectID].append({
                    'frame': frame_num,
                    'bbox': bbox,
                    'class': class_names[col]
                })

                used_rows.add(row)
                used_cols.add(col)

            # Objets non associés
            unused_rows = set(range(0, D.shape[0])).difference(used_rows)
            unused_cols = set(range(0, D.shape[1])).difference(used_cols)

            if D.shape[0] >= D.shape[1]:
                for row in unused_rows:
                    objectID = objectIDs[row]
                    self.disappeared[objectID] += 1
                    if self.disappeared[objectID] > self.maxDisappeared:
                        self.deregister(objectID)
            else:
                for col in unused_cols:
                    self.register(input_centroids[col], class_names[col])
                    bbox = rects[col]
                    self.vehicle_history[self.nextObjectID - 1].append({
                        'frame': frame_num,
                        'bbox': bbox,
                        'class': class_names[col]
                    })

        return self.objects

    def get_completed_vehicles(self):
        """Retourner les véhicules complètement suivis (disparus depuis longtemps)"""
        return self.vehicle_history

In [ ]:
# ============================================================================
# Fonctions de détection et comptage des véhicules
# ============================================================================

def find_best_yolo_model(base_path='vehicle-detection'):
    """
    Trouver le meilleur modèle YOLO dans le répertoire
    """
    model_dirs = glob(os.path.join(base_path, 'yolov8n-*'))
    if not model_dirs:
        print(f"❌ Aucun modèle trouvé dans {base_path}")
        return None
    
    # Retourner le dernier modèle (supposément le meilleur)
    latest_model_dir = sorted(model_dirs)[-1]
    weights_path = os.path.join(latest_model_dir, 'weights', 'best.pt')
    
    if os.path.exists(weights_path):
        print(f"✅ Modèle trouvé: {weights_path}")
        return weights_path
    return None

def load_yolo_model(model_path):
    """Charger le modèle YOLO"""
    try:
        model = YOLO(model_path)
        print(f"✅ Modèle YOLO chargé avec succès")
        return model
    except Exception as e:
        print(f"❌ Erreur lors du chargement du modèle: {e}")
        return None

def process_video_for_vehicle_counting(video_path, model, tracker, 
                                       conf_threshold=0.5, 
                                       class_mapping={0: 'car', 1: 'bus', 2: 'van'},
                                       output_video_path=None,
                                       show_progress=True):
    """
    Traiter une vidéo complète pour compter et suivre les véhicules
    
    Args:
        video_path: Chemin vers le fichier vidéo
        model: Modèle YOLO chargé
        tracker: Instance de CentroidTracker
        conf_threshold: Seuil de confiance pour les détections
        class_mapping: Mapping des IDs de classe aux noms
        output_video_path: Chemin pour sauvegarder la vidéo annotée (optionnel)
        show_progress: Afficher la barre de progression
    
    Returns:
        Dict avec les résultats du comptage
    """
    
    # Ouvrir la vidéo
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Impossible d'ouvrir la vidéo: {video_path}")
        return None
    
    # Propriétés de la vidéo
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"\n📹 Vidéo: {os.path.basename(video_path)}")
    print(f"   Résolution: {width}x{height}, FPS: {fps}, Total frames: {total_frames}")
    
    # Initialiser le writer vidéo si nécessaire
    out = None
    if output_video_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    # Traitement des frames
    frame_num = 0
    detections_per_frame = []
    
    pbar = tqdm(total=total_frames, desc="Traitement vidéo") if show_progress else None
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Détecter les véhicules avec YOLO
        results = model(frame, conf=conf_threshold, verbose=False)
        
        # Extraire les détections
        rects = []
        class_names = []
        confidences = []
        
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])
                
                # Filtrer par classe (0, 1, 2 = car, bus, van)
                if cls_id in class_mapping:
                    rects.append((x1, y1, x2, y2))
                    class_names.append(class_mapping[cls_id])
                    confidences.append(conf)
        
        # Mettre à jour le tracker
        tracked_objects = tracker.update(rects, class_names, frame_num)
        
        # Enregistrer les détections du frame
        detections_per_frame.append({
            'frame': frame_num,
            'detections': len(rects),
            'tracked_ids': list(tracked_objects.keys())
        })
        
        # Annoter le frame si nécessaire
        if output_video_path or show_progress:
            annotated_frame = frame.copy()
            
            # Dessiner les boîtes détectées et les IDs de suivi
            for objectID, centroid in tracked_objects.items():
                if objectID < len(tracker.vehicle_history):
                    # Récupérer la dernière bbox de cet objet
                    if objectID in tracker.vehicle_history and len(tracker.vehicle_history[objectID]) > 0:
                        last_entry = tracker.vehicle_history[objectID][-1]
                        bbox = last_entry['bbox']
                        class_name = last_entry['class']
                        x1, y1, x2, y2 = bbox
                        
                        # Dessiner le rectangle
                        color = (0, 255, 0) if class_name == 'car' else (0, 165, 255) if class_name == 'bus' else (255, 0, 0)
                        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                        
                        # Dessiner l'ID et la classe
                        label = f"ID:{objectID} {class_name}"
                        cv2.putText(annotated_frame, label, (x1, y1 - 10), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
            if output_video_path:
                out.write(annotated_frame)
        
        frame_num += 1
        if pbar:
            pbar.update(1)
    
    if pbar:
        pbar.close()
    
    cap.release()
    if out:
        out.release()
    
    # Compiler les résultats
    vehicle_counts = {'car': 0, 'bus': 0, 'van': 0}
    vehicle_details = []
    
    for vehicle_id, history in tracker.vehicle_history.items():
        if len(history) > 0:
            first_frame = history[0]['frame']
            last_frame = history[-1]['frame']
            duration = last_frame - first_frame + 1
            vehicle_class = history[0]['class']
            
            if vehicle_class in vehicle_counts:
                vehicle_counts[vehicle_class] += 1
            
            # Calculer la position moyenne
            bboxes = [h['bbox'] for h in history]
            avg_x = np.mean([((b[0] + b[2]) // 2) for b in bboxes])
            avg_y = np.mean([((b[1] + b[3]) // 2) for b in bboxes])
            
            vehicle_details.append({
                'id': vehicle_id,
                'class': vehicle_class,
                'first_frame': first_frame,
                'last_frame': last_frame,
                'duration_frames': duration,
                'avg_x': int(avg_x),
                'avg_y': int(avg_y)
            })
    
    results_dict = {
        'video_path': video_path,
        'video_name': os.path.basename(video_path),
        'fps': fps,
        'width': width,
        'height': height,
        'total_frames': total_frames,
        'vehicle_counts': vehicle_counts,
        'vehicle_details': vehicle_details,
        'total_vehicles': sum(vehicle_counts.values()),
        'processing_timestamp': datetime.now().isoformat()
    }
    
    return results_dict

In [ ]:
# ============================================================================
# Fonctions de visualisation et résultats
# ============================================================================

def display_counting_results(results):
    """Afficher les résultats du comptage de véhicules"""
    
    print("\n" + "="*70)
    print(f"📊 RÉSULTATS DE COMPTAGE - {results['video_name']}")
    print("="*70)
    
    print(f"\n📹 Vidéo:")
    print(f"  • Résolution: {results['width']}x{results['height']}")
    print(f"  • FPS: {results['fps']}")
    print(f"  • Total frames: {results['total_frames']}")
    print(f"  • Durée approximative: {results['total_frames']/results['fps']:.1f}s")
    
    print(f"\n🚗 COMPTAGE PAR TYPE DE VÉHICULE:")
    print(f"  • Voitures (car):  {results['vehicle_counts']['car']:3d} véhicules")
    print(f"  • Bus:             {results['vehicle_counts']['bus']:3d} véhicules")
    print(f"  • Camionnettes:    {results['vehicle_counts']['van']:3d} véhicules")
    print(f"  {'─'*40}")
    print(f"  • TOTAL:           {results['total_vehicles']:3d} véhicules")
    
    print(f"\n📋 DÉTAILS DES VÉHICULES DÉTECTÉS:")
    print(f"{'ID':<5} {'Type':<10} {'Frames':<20} {'Durée':<10}")
    print(f"{'─'*50}")
    
    for detail in sorted(results['vehicle_details'], key=lambda x: x['first_frame']):
        duration = f"{detail['duration_frames']}"
        frames_range = f"{detail['first_frame']}-{detail['last_frame']}"
        print(f"{detail['id']:<5} {detail['class']:<10} {frames_range:<20} {duration:<10}")
    
    print("="*70 + "\n")
    
    return results

def plot_counting_statistics(results_list):
    """Créer des graphiques de statistiques"""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Graphique 1: Comptage par vidéo
    video_names = [r['video_name'][:20] for r in results_list]  # Tronquer les noms longs
    car_counts = [r['vehicle_counts']['car'] for r in results_list]
    bus_counts = [r['vehicle_counts']['bus'] for r in results_list]
    van_counts = [r['vehicle_counts']['van'] for r in results_list]
    
    x = np.arange(len(video_names))
    width = 0.25
    
    axes[0, 0].bar(x - width, car_counts, width, label='Voitures', color='#2ecc71')
    axes[0, 0].bar(x, bus_counts, width, label='Bus', color='#e74c3c')
    axes[0, 0].bar(x + width, van_counts, width, label='Camionnettes', color='#3498db')
    axes[0, 0].set_xlabel('Vidéo')
    axes[0, 0].set_ylabel('Nombre de véhicules')
    axes[0, 0].set_title('Comptage par vidéo')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(video_names, rotation=45, ha='right')
    axes[0, 0].legend()
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # Graphique 2: Répartition globale par type
    total_cars = sum(car_counts)
    total_buses = sum(bus_counts)
    total_vans = sum(van_counts)
    
    colors = ['#2ecc71', '#e74c3c', '#3498db']
    sizes = [total_cars, total_buses, total_vans]
    labels = [f'Voitures\n({total_cars})', f'Bus\n({total_buses})', f'Camionnettes\n({total_vans})']
    
    axes[0, 1].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    axes[0, 1].set_title('Répartition globale des véhicules')
    
    # Graphique 3: Total par vidéo
    totals = [r['total_vehicles'] for r in results_list]
    axes[1, 0].bar(range(len(video_names)), totals, color='#9b59b6')
    axes[1, 0].set_xlabel('Vidéo')
    axes[1, 0].set_ylabel('Total véhicules')
    axes[1, 0].set_title('Total véhicules par vidéo')
    axes[1, 0].set_xticks(range(len(video_names)))
    axes[1, 0].set_xticklabels(video_names, rotation=45, ha='right')
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # Graphique 4: Durée moyenne des véhicules
    all_durations = []
    all_classes = []
    for results in results_list:
        for detail in results['vehicle_details']:
            all_durations.append(detail['duration_frames'])
            all_classes.append(detail['class'])
    
    if all_durations:
        car_durations = [d for d, c in zip(all_durations, all_classes) if c == 'car']
        bus_durations = [d for d, c in zip(all_durations, all_classes) if c == 'bus']
        van_durations = [d for d, c in zip(all_durations, all_classes) if c == 'van']
        
        duration_data = [car_durations, bus_durations, van_durations]
        axes[1, 1].boxplot(duration_data, labels=['Voitures', 'Bus', 'Camionnettes'])
        axes[1, 1].set_ylabel('Durée (frames)')
        axes[1, 1].set_title('Distribution des durées de présence')
        axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def save_results_to_json(results_list, output_path='vehicle_counting_results.json'):
    """Sauvegarder les résultats en JSON"""
    with open(output_path, 'w') as f:
        json.dump(results_list, f, indent=2)
    print(f"✅ Résultats sauvegardés: {output_path}")

def save_results_to_csv(results_list, output_path='vehicle_counting_results.csv'):
    """Sauvegarder les résultats en CSV"""
    data = []
    for results in results_list:
        for detail in results['vehicle_details']:
            data.append({
                'video': results['video_name'],
                'vehicle_id': detail['id'],
                'type': detail['class'],
                'first_frame': detail['first_frame'],
                'last_frame': detail['last_frame'],
                'duration_frames': detail['duration_frames'],
                'avg_x': detail['avg_x'],
                'avg_y': detail['avg_y']
            })
    
    df = pd.DataFrame(data)
    df.to_csv(output_path, index=False)
    print(f"✅ Résultats CSV sauvegardés: {output_path}")

## Exécution complète du comptage

In [ ]:
INPUT_PATH = os.path.join(os.getcwd(), 'data')
TRAIN_ANNOTATIONS_DIR = os.path.join(INPUT_PATH, 'DETRAC-Train-Annotations-XML', 'DETRAC-Train-Annotations-XML')
TEST_ANNOTATIONS_DIR = os.path.join(INPUT_PATH, 'DETRAC-Test-Annotations-XML', 'DETRAC-Test-Annotations-XML')
ALL_IMAGES_DIR = os.path.join(INPUT_PATH, 'DETRAC-Images', 'DETRAC-Images')

In [ ]:
# Configuration
VIDEO_DIR = 'data/video'
MODEL_BASE_DIR = 'vehicle-detection'
OUTPUT_DIR = 'vehicle_counting_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Charger le modèle YOLO
model_path = find_best_yolo_model(MODEL_BASE_DIR)
if model_path is None:
    print("⚠️ Modèle non trouvé. Assurez-vous que le modèle YOLO est présent.")
    model = None
else:
    model = load_yolo_model(model_path)

# Classe mapping
CLASS_MAPPING = {0: 'car', 1: 'bus', 2: 'van'}

print(f"\n📂 Dossier des vidéos: {VIDEO_DIR}")
print(f"📂 Dossier de sortie: {OUTPUT_DIR}")

In [ ]:
# Trouver toutes les vidéos à traiter
video_extensions = ['*.avi', '*.mp4', '*.mov', '*.mkv']
video_files = []

for ext in video_extensions:
    video_files.extend(glob(os.path.join(VIDEO_DIR, ext)))

print(f"\n🎬 Vidéos trouvées: {len(video_files)}")
for vid in video_files[:5]:  # Afficher les 5 premières
    print(f"   • {os.path.basename(vid)}")
if len(video_files) > 5:
    print(f"   ... et {len(video_files) - 5} autres")

# Traiter chaque vidéo
all_results = []

if model is not None:
    for video_path in video_files:
        # Créer un tracker pour cette vidéo
        tracker = CentroidTracker(maxDisappeared=30, maxDistance=50)
        
        # Chemin pour la vidéo de sortie
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        output_video = os.path.join(OUTPUT_DIR, f'{video_name}_tracked.mp4')
        
        # Traiter la vidéo
        results = process_video_for_vehicle_counting(
            video_path=video_path,
            model=model,
            tracker=tracker,
            conf_threshold=0.5,
            class_mapping=CLASS_MAPPING,
            output_video_path=output_video,
            show_progress=True
        )
        
        if results:
            all_results.append(results)
            display_counting_results(results)
else:
    print("⚠️ Modèle YOLO non disponible. Impossible de traiter les vidéos.")
    print("Assurez-vous que le modèle est présent dans 'vehicle-detection/'")


In [ ]:
# Sauvegarder et visualiser les résultats
if all_results:
    print("\n" + "="*70)
    print("💾 SAUVEGARDE DES RÉSULTATS")
    print("="*70)
    
    # Sauvegarder en JSON
    save_results_to_json(all_results, os.path.join(OUTPUT_DIR, 'results.json'))
    
    # Sauvegarder en CSV
    save_results_to_csv(all_results, os.path.join(OUTPUT_DIR, 'results.csv'))
    
    # Afficher les graphiques
    print("\n📊 Génération des graphiques statistiques...")
    plot_counting_statistics(all_results)
    
    # Afficher un résumé global
    print("\n" + "="*70)
    print("📈 RÉSUMÉ GLOBAL")
    print("="*70)
    
    total_videos = len(all_results)
    total_vehicles = sum(r['total_vehicles'] for r in all_results)
    total_cars = sum(r['vehicle_counts']['car'] for r in all_results)
    total_buses = sum(r['vehicle_counts']['bus'] for r in all_results)
    total_vans = sum(r['vehicle_counts']['van'] for r in all_results)
    
    print(f"\n📹 Vidéos traitées: {total_videos}")
    print(f"🚗 Total véhicules comptés: {total_vehicles}")
    print(f"   • Voitures: {total_cars} ({100*total_cars/total_vehicles:.1f}%)")
    print(f"   • Bus: {total_buses} ({100*total_buses/total_vehicles:.1f}%)")
    print(f"   • Camionnettes: {total_vans} ({100*total_vans/total_vehicles:.1f}%)")
    
    print(f"\n✅ Résultats sauvegardés dans: {os.path.abspath(OUTPUT_DIR)}")
else:
    print("\n⚠️ Aucun résultat à afficher.")